# 01 — Data Layer (Evening 1)

**Research questions answered here:**
- **RQ1**: Can we get reliable, point-in-time NSE OHLCV data?
- **RQ2**: Can Python (not LLM) compute the required indicators?

**Gate before moving to Evening 2:**
- ATR(14) for INFY matches TradingView within 2%
- EMA-20 matches TradingView within 0.5%
- No NaN in OHLCV columns
- Swing levels are real price points from the chart

**Cost: ₹0** — no LLM calls in this notebook.

In [1]:
# Cell 1 — Imports
import warnings; warnings.filterwarnings('ignore')
from dotenv import load_dotenv
import os, sys
from datetime import date, timedelta

import pandas as pd
import pandas_ta as ta
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

load_dotenv('../.env') or load_dotenv('.env')

# Use today as the as_of_date for live POC
# Change to a historical date for point-in-time testing, e.g. '2025-09-15'
AS_OF_DATE = date.today().isoformat()
print(f'as_of_date: {AS_OF_DATE}')

as_of_date: 2026-06-11


## Step 1 — Fetch OHLCV (RQ1)

Function defined inline for now. We'll extract to `src/research/data.py` once it's validated.

In [2]:
# Cell 2 — get_ohlcv: fetch + clean + point-in-time slice
# Spec ref: §002 data-layer plan.md, §20.3 MVP scope

def get_ohlcv(symbol: str, as_of_date: str) -> pd.DataFrame:
    """
    Fetch NSE OHLCV via yfinance up to and including as_of_date.
    Returns clean DataFrame with lowercase columns, date index.
    Index symbols (starting with ^) are used as-is; equity symbols get .NS appended.
    Raises ValueError if data is empty or insufficient.
    """
    ticker = symbol if symbol.startswith('^') else f'{symbol}.NS'
    raw = yf.download(ticker, period='max', auto_adjust=True, progress=False)

    if raw.empty:
        raise ValueError(f'No data for {ticker} — check symbol or internet')

    # Flatten MultiIndex columns (yfinance quirk for single tickers)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.droplevel(1)

    # Normalise column names
    raw.columns = [c.lower().replace(' ', '_') for c in raw.columns]

    # Convert index to plain date (no timezone)
    raw.index = pd.to_datetime(raw.index).normalize().date

    # --- Point-in-time lockout: no data after as_of_date ---
    cutoff = date.fromisoformat(as_of_date)
    df = raw[raw.index <= cutoff].copy()

    if len(df) < 100:
        raise ValueError(f'{symbol} has only {len(df)} rows — need 100+ for indicators')

    return df


# Test it
df_infy = get_ohlcv('INFY', AS_OF_DATE)
print(f'INFY: {len(df_infy)} rows  |  last date: {df_infy.index[-1]}  |  cols: {list(df_infy.columns)}')
df_infy.tail(3)

INFY: 7643 rows  |  last date: 2026-06-11  |  cols: ['close', 'high', 'low', 'open', 'volume']


,close,high,low,open,volume
2026-06-09,1155.300049,1168.318269,1142.575403,1168.318269,10984054
2026-06-10,1145.300049,1171.400024,1143.300049,1160.099976,8887626
2026-06-11,1119.000000,1133.699951,1109.000000,1129.699951,4130100


In [3]:
# Cell 3 — Data quality checks (RQ1 gate)

def check_data_quality(df: pd.DataFrame, symbol: str) -> bool:
    """Assert data is clean. Print result per check."""
    passed = True

    # Check 1: no NaN in price columns
    price_cols = ['open','high','low','close','volume']
    nan_count = df[price_cols].isnull().sum().sum()
    if nan_count == 0:
        print(f'  ✓ No NaN in OHLCV columns')
    else:
        print(f'  ✗ {nan_count} NaN found in OHLCV — investigate')
        passed = False

    # Check 2: volume > 0
    zero_vol = (df['volume'] == 0).sum()
    if zero_vol == 0:
        print(f'  ✓ All volume > 0')
    else:
        print(f'  ⚠  {zero_vol} zero-volume rows (likely holidays — OK to ignore)')

    # Check 3: prices are positive
    if (df['close'] > 0).all():
        print(f'  ✓ All close prices > 0')
    else:
        print(f'  ✗ Non-positive close prices found')
        passed = False

    # Check 4: data is recent (last date within 5 trading days of today)
    last_date = df.index[-1]
    today = date.today()
    days_stale = (today - last_date).days
    if days_stale <= 5:
        print(f'  ✓ Data is fresh  (last: {last_date}, {days_stale}d ago)')
    else:
        print(f'  ⚠  Data last updated {days_stale} days ago — may be stale')

    status = '✓ PASS' if passed else '✗ FAIL'
    print(f'\n  Data quality {symbol}: {status}')
    return passed


print('INFY data quality:')
check_data_quality(df_infy, 'INFY')

INFY data quality:
  ✓ No NaN in OHLCV columns
  ⚠  124 zero-volume rows (likely holidays — OK to ignore)
  ✓ All close prices > 0
  ✓ Data is fresh  (last: 2026-06-11, 0d ago)

  Data quality INFY: ✓ PASS


True

## Step 2 — Compute indicators (RQ2)


In [4]:
# Cell 4 — compute_indicators: all indicators used by the Technical Analyst
# Spec ref: §4 Agent 2 computations table, constitution §domain rules
# IMPORTANT: indicators are ALWAYS computed in Python — never by LLM

def compute_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add all technical indicators to OHLCV DataFrame.
    Pure Python / pandas-ta — no LLM calls.
    Modifies a copy; does not alter the input DataFrame.
    """
    df = df.copy()

    # Trend EMAs
    df['ema_20']  = ta.ema(df['close'], length=20)
    df['ema_50']  = ta.ema(df['close'], length=50)
    df['ema_200'] = ta.ema(df['close'], length=200)

    # Volatility
    df['atr_14']  = ta.atr(df['high'], df['low'], df['close'], length=14)

    # Momentum
    df['rsi_14']  = ta.rsi(df['close'], length=14)

    # Bollinger Bands — use .filter() to handle column name changes across pandas_ta versions
    bb = ta.bbands(df['close'], length=20, std=2.0)
    if bb is not None:
        upper_col = bb.filter(like='BBU').columns[0]
        lower_col = bb.filter(like='BBL').columns[0]
        df['bb_upper'] = bb[upper_col]
        df['bb_lower'] = bb[lower_col]

    # Volume average
    df['vol_avg_20'] = df['volume'].rolling(20).mean()

    return df


df_infy = compute_indicators(df_infy)

# Show last row — include volume so last.volume is accessible
indicator_cols = ['close','volume','ema_20','ema_50','ema_200','atr_14','rsi_14','bb_upper','bb_lower','vol_avg_20']
last = df_infy[indicator_cols].iloc[-1]

print('INFY indicators (most recent close):')
print(f'  Close:      ₹{last.close:.2f}')
print(f'  EMA-20:     ₹{last.ema_20:.2f}  ({"above" if last.close > last.ema_20 else "below"})')
print(f'  EMA-50:     ₹{last.ema_50:.2f}  ({"above" if last.close > last.ema_50 else "below"})')
print(f'  EMA-200:    ₹{last.ema_200:.2f} ({"above" if last.close > last.ema_200 else "below"})')
print(f'  ATR(14):    ₹{last.atr_14:.2f}  ({last.atr_14/last.close*100:.1f}% of price)')
print(f'  RSI(14):    {last.rsi_14:.1f}')
print(f'  Volume:     {last.volume:,.0f}  ({last.volume/last.vol_avg_20*100:.0f}% of 20-day avg)')
print()
print('>>> Open TradingView → INFY → compare ATR(14) and EMA-20 manually')
print(f'>>> TradingView link: https://www.tradingview.com/chart/?symbol=NSE:INFY')

INFY indicators (most recent close):
  Close:      ₹1119.00
  EMA-20:     ₹1157.94  (below)
  EMA-50:     ₹1190.43  (below)
  EMA-200:    ₹1351.50 (below)
  ATR(14):    ₹32.18  (2.9% of price)
  RSI(14):    40.9


AttributeError: 'Series' object has no attribute 'volume'

In [ ]:
# Cell 5 — TradingView comparison check
# After running Cell 4, open TradingView and compare these values
# Enter what TradingView shows here:

TV_ATR_14  = None  # e.g. 31.4  — paste from TradingView
TV_EMA_20  = None  # e.g. 1482.3

our_atr = df_infy['atr_14'].iloc[-1]
our_ema = df_infy['ema_20'].iloc[-1]

print(f'Our ATR(14):   ₹{our_atr:.2f}')
print(f'Our EMA-20:    ₹{our_ema:.2f}')
print()

if TV_ATR_14 is not None:
    atr_diff = abs(our_atr - TV_ATR_14) / TV_ATR_14 * 100
    gate = '✓ PASS' if atr_diff <= 2.0 else '✗ FAIL'
    print(f'ATR diff: {atr_diff:.2f}%  {gate}  (gate: <= 2%)')
else:
    print('⚠  Set TV_ATR_14 above after checking TradingView — this is the RQ2 gate')

if TV_EMA_20 is not None:
    ema_diff = abs(our_ema - TV_EMA_20) / TV_EMA_20 * 100
    gate = '✓ PASS' if ema_diff <= 0.5 else '✗ FAIL'
    print(f'EMA-20 diff: {ema_diff:.2f}%  {gate}  (gate: <= 0.5%)')

## Step 3 — Swing levels (ZigZag)


In [ ]:
# Cell 6 — identify_swings: ZigZag 3% swing high/low detection
# Returns nearest support and resistance to last close

def identify_swings(df: pd.DataFrame, pct: float = 0.03, lookback: int = 500) -> dict:
    """
    Find swing highs and lows using ZigZag (3% threshold).
    Operates on the most recent `lookback` bars to avoid ancient price levels
    distorting support/resistance for current price action.
    Returns nearest support and resistance to the last close.
    All levels are real prices from the data — no round numbers.
    """
    # Limit to recent price history
    df = df.iloc[-lookback:].copy()

    highs, lows = [], []
    hi_list = df['high'].values
    lo_list = df['low'].values
    n = len(df)

    direction = None   # 'up' or 'down'
    last_pivot = df['close'].iloc[0]

    for i in range(1, n):
        h, l = hi_list[i], lo_list[i]
        if direction != 'down' and h >= last_pivot * (1 + pct):
            highs.append(round(float(h), 2))
            last_pivot = h
            direction = 'down'
        elif direction != 'up' and l <= last_pivot * (1 - pct):
            lows.append(round(float(l), 2))
            last_pivot = l
            direction = 'up'

    last_close = float(df['close'].iloc[-1])

    # Nearest levels relative to current price
    supports    = [l for l in lows  if l < last_close]
    resistances = [h for h in highs if h > last_close]

    nearest_support    = max(supports,    default=round(float(df['low'].min()), 2))
    nearest_resistance = min(resistances, default=round(float(df['high'].max()), 2))

    recent_highs = sorted(highs, reverse=True)[:5]
    recent_lows  = sorted(lows,  reverse=True)[:5]

    return {
        'nearest_support':    nearest_support,
        'nearest_resistance': nearest_resistance,
        'recent_highs':       recent_highs,
        'recent_lows':        recent_lows,
    }


swings = identify_swings(df_infy)
last_close = df_infy['close'].iloc[-1]

print(f'INFY swing levels  (last close: ₹{last_close:.2f}):')
print(f'  Nearest support:    ₹{swings["nearest_support"]}')
print(f'  Nearest resistance: ₹{swings["nearest_resistance"]}')
print(f'  Recent highs:       {swings["recent_highs"]}')
print(f'  Recent lows:        {swings["recent_lows"]}')
print()
assert swings['nearest_support'] < last_close < swings['nearest_resistance'], \
    'Support/resistance logic error'
print('✓ Support < close < Resistance')

## Step 4 — Relative strength vs Nifty500


In [ ]:
# Cell 7 — relative_strength: 13-week RS percentile
# Spec ref: §4 Agent 2 computations, constitution §scoring formula
# rs_rating_pct is a SEPARATE field from technical_score (v1.2 fix)
#
# Benchmark: ^NSEI (Nifty 50) — Nifty 500 (^CNX500) is not available on yfinance

def relative_strength(symbol: str, as_of_date: str, window_weeks: int = 13) -> float:
    """
    Compute RS of stock vs Nifty 50 as 0-100 percentile.
    13-week (65 trading day) return ratio.
    Returns 0-100 where 100 = strongest outperformer.
    """
    window_days = window_weeks * 5

    stock_df = get_ohlcv(symbol, as_of_date)
    nifty_df = get_ohlcv('^NSEI', as_of_date)   # Nifty 50 — reliable via yfinance

    if len(stock_df) < window_days or len(nifty_df) < window_days:
        return 50.0  # neutral fallback if insufficient history

    stock_ret = stock_df['close'].iloc[-1] / stock_df['close'].iloc[-window_days] - 1
    nifty_ret = nifty_df['close'].iloc[-1] / nifty_df['close'].iloc[-window_days] - 1

    # RS ratio > 1.0 means outperforming
    rs_ratio   = (1 + stock_ret) / (1 + nifty_ret)

    # Map to 0-100 percentile using a reasonable range (0.7 to 1.3)
    percentile = min(100.0, max(0.0, (rs_ratio - 0.7) / (1.3 - 0.7) * 100))
    return round(percentile, 1)


rs_infy = relative_strength('INFY', AS_OF_DATE)
print(f'INFY RS vs Nifty 50 (13-week): {rs_infy:.1f}th percentile')
print()
if rs_infy >= 70:
    print('  → Strong RS leader')
elif rs_infy >= 50:
    print('  → Above average — qualifies for consideration')
else:
    print('  → Below average — may be rejected by Technical Analyst')

## Step 5 — Charts (the visual RQ1 check)


In [ ]:
# Cell 8 — Chart: candlestick + EMAs + swing levels for one stock

def plot_stock(symbol: str, as_of_date: str, days: int = 60) -> None:
    """Plot candlestick with EMAs and swing levels. Renders inline in Jupyter."""
    df = get_ohlcv(symbol, as_of_date)
    df = compute_indicators(df)
    swings = identify_swings(df)
    rs = relative_strength(symbol, as_of_date)

    df_plot = df.iloc[-days:]
    x = [str(d) for d in df_plot.index]

    fig = make_subplots(
        rows=2, cols=1,
        row_heights=[0.75, 0.25],
        shared_xaxes=True,
        vertical_spacing=0.03,
    )

    # Candlestick
    fig.add_trace(go.Candlestick(
        x=x, open=df_plot['open'], high=df_plot['high'],
        low=df_plot['low'],   close=df_plot['close'],
        name='OHLC', increasing_line_color='#26a69a',
        decreasing_line_color='#ef5350',
    ), row=1, col=1)

    # EMAs
    for col, color, name in [
        ('ema_20',  '#FF9800', 'EMA-20'),
        ('ema_50',  '#2196F3', 'EMA-50'),
        ('ema_200', '#9C27B0', 'EMA-200'),
    ]:
        fig.add_trace(go.Scatter(
            x=x, y=df_plot[col], name=name,
            line=dict(color=color, width=1.5),
        ), row=1, col=1)

    # Swing levels as horizontal lines
    fig.add_hline(y=swings['nearest_support'],    line_color='green',
                  line_dash='dot',  annotation_text=f'S ₹{swings["nearest_support"]}',
                  annotation_position='left', row=1, col=1)
    fig.add_hline(y=swings['nearest_resistance'], line_color='red',
                  line_dash='dot',  annotation_text=f'R ₹{swings["nearest_resistance"]}',
                  annotation_position='left', row=1, col=1)

    # Volume
    colors = ['#26a69a' if c >= o else '#ef5350'
              for c, o in zip(df_plot['close'], df_plot['open'])]
    fig.add_trace(go.Bar(
        x=x, y=df_plot['volume'], name='Volume',
        marker_color=colors, showlegend=False,
    ), row=2, col=1)
    fig.add_trace(go.Scatter(
        x=x, y=df_plot['vol_avg_20'], name='Vol avg',
        line=dict(color='orange', width=1), showlegend=False,
    ), row=2, col=1)

    last = df.iloc[-1]
    title = (f'{symbol}  |  {as_of_date}  |  '
             f'ATR(14): ₹{last.atr_14:.1f}  |  '
             f'RSI: {last.rsi_14:.0f}  |  '
             f'RS: {rs:.0f}th pct')

    fig.update_layout(
        title=title, height=480,
        xaxis_rangeslider_visible=False,
        template='plotly_dark',
        legend=dict(orientation='h', yanchor='bottom', y=1.01),
        margin=dict(l=60, r=60, t=60, b=40),
    )
    fig.show()


plot_stock('INFY', AS_OF_DATE)

In [ ]:
# Cell 9 — Run all 5 POC stocks
# These cover different sectors for a representative sample

POC_STOCKS = [
    'INFY',       # Nifty IT — software services
    'RELIANCE',   # Large cap conglomerate
    'DRREDDY',    # Nifty Pharma
    'LT',         # Nifty Infra
    'HDFCBANK',   # Nifty Banking
]

print(f'Running data quality + indicator checks for {len(POC_STOCKS)} stocks...\n')
results = []

for sym in POC_STOCKS:
    try:
        df_sym = get_ohlcv(sym, AS_OF_DATE)
        df_sym = compute_indicators(df_sym)
        sw     = identify_swings(df_sym)
        rs     = relative_strength(sym, AS_OF_DATE)
        last   = df_sym.iloc[-1]

        results.append({
            'Symbol': sym,
            'Close':  f'₹{last.close:.1f}',
            'ATR(14)': f'₹{last.atr_14:.1f} ({last.atr_14/last.close*100:.1f}%)',
            'RSI':    f'{last.rsi_14:.0f}',
            'EMA-20': f'₹{last.ema_20:.1f}',
            'vs EMA-20': 'above' if last.close > last.ema_20 else 'below',
            'RS pct': f'{rs:.0f}th',
            'Support': f'₹{sw["nearest_support"]}',
            'Resistance': f'₹{sw["nearest_resistance"]}',
            'Rows': len(df_sym),
        })
        print(f'  ✓ {sym}')
    except Exception as e:
        print(f'  ✗ {sym}: {e}')
        results.append({'Symbol': sym, 'Error': str(e)})

print()
display(pd.DataFrame(results))

In [ ]:
# Cell 10 — Chart all 5 stocks
for sym in POC_STOCKS:
    try:
        plot_stock(sym, AS_OF_DATE)
    except Exception as e:
        print(f'{sym}: {e}')

## Evening 1 Gate

Before moving to `02_technical_agent.ipynb`:

- [ ] ATR(14) for INFY matches TradingView within **2%** (update `TV_ATR_14` in Cell 5)
- [ ] EMA-20 for INFY matches TradingView within **0.5%**
- [ ] Charts look like real candlestick charts with correct EMA positions
- [ ] Support and resistance levels are at real swing points — not round numbers
- [ ] All 5 stocks return data without errors

If all pass → open `02_technical_agent.ipynb` for Evening 2.